## Board Diversity and Firm Performance in Publicly Listed Technology Companies
#### Study Period: 2009–2018

#### Sample: 20 publicly listed technology companies

#### Observations: 200 firm-year observations

In [6]:
import pandas as pd
import numpy as np
import scipy
import statsmodels.api as sm

print("Everything imported successfully!")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Statsmodels:", sm.__version__)

Everything imported successfully!
Pandas: 3.0.1
NumPy: 2.4.4
SciPy: 1.18.0
Statsmodels: 0.14.6


In [7]:
import os

print(os.getcwd())
print(os.listdir())

C:\Users\KARTHIK RAJ S
['.anaconda', '.conda', '.condarc', '.continuum', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.virtual_documents', '1942 Assignment 2', '1943 Final report (2).ipynb', 'anaconda_projects', 'AppData', 'Application Data', 'Assignment 2 Template.ipynb', 'Board_Diversity_Analysis.ipynb', 'Contacts', 'Cookies', 'Coupon_Recommendation.csv', 'Dataset_partA.xlsx', 'Dataset_partB.xlsx', 'Desktop', 'Documents', 'Downloads', 'Favorites', 'Final_Correlation_Matrix.csv', 'Final_Descriptive_Statistics.csv', 'Final_Diagnostics.csv', 'Final_Locked_Technology_Dataset_200_Observations(5).csv', 'Final_Model_Summary.csv', 'Final_Regression_Results_HC3.csv', 'Final_VIF.csv', 'Links', 'Local Settings', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{b276d4a2-24aa-11f0-adb3-f83dc60729d5}.TM.blf', 'NTUSER.DAT{b276d4a2-24aa-11f0-adb3-f83dc60729d5}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{b276d4a2-24aa-1

In [8]:
df = pd.read_csv("Final_Locked_Technology_Dataset_200_Observations(5).csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()

Dataset loaded successfully!
Dataset shape: (200, 11)


,Company,Year,SIC Code,Female Directors (%),Female Directors (N),Board Size,Independent Directors (N),Board Independence (%),ROA (%),Firm Size (ln assets),Leverage (%)
0,Adobe Inc,2009,7372,10.00000,1,10,7,70.00000,10.04874,8.893193,32.842504
1,Adobe Inc,2010,7372,10.00000,1,10,7,70.00000,12.48399,9.004686,36.220457
2,Adobe Inc,2011,7372,0.00000,0,9,6,66.66667,13.31384,9.104000,35.680177
3,Adobe Inc,2012,7372,16.66667,2,12,9,75.00000,11.80281,9.207789,33.177937
4,Adobe Inc,2013,7372,15.38462,2,13,10,76.92308,4.32762,9.247664,35.217332


#### Check Dataset Structure

In [9]:

print("Number of companies:", df["Company"].nunique())
print("Year range:", df["Year"].min(), "to", df["Year"].max())

print("\nMissing values:")
print(df.isnull().sum())

print("\nObservations per company:")
print(df.groupby("Company").size())

Number of companies: 20
Year range: 2009 to 2018

Missing values:
Company                      0
Year                         0
SIC Code                     0
Female Directors (%)         0
Female Directors (N)         0
Board Size                   0
Independent Directors (N)    0
Board Independence (%)       0
ROA (%)                      0
Firm Size (ln assets)        0
Leverage (%)                 0
dtype: int64

Observations per company:
Company
Adobe Inc                  10
Akamai Technologies Inc    10
Analog Devices             10
Apple Inc                  10
Applied Materials Inc      10
Broadcom Inc               10
Cisco Systems Inc          10
Ebay Inc                   10
Intel Corp                 10
Intuit Inc                 10
Juniper Networks Inc       10
Kla-Tencor Corp            10
Lam Research Corp          10
Micron Technology Inc      10
Microsoft Corp             10
Netapp Inc                 10
Nvidia Corp                10
Oracle Corp                10
Qualc

####  Check Board Independence values above 100%

In [10]:
independence_check = df[df["Board Independence (%)"] > 100][
    ["Company", "Year", "Board Size",
     "Independent Directors (N)", "Board Independence (%)"]
]

print("Rows with Board Independence above 100%:")
display(independence_check)

Rows with Board Independence above 100%:


,Company,Year,Board Size,Independent Directors (N),Board Independence (%)
26,Analog Devices,2015,11,12,109.0909
27,Analog Devices,2016,10,12,120.0000


In [11]:
import pandas as pd

df = pd.read_csv("Final_Locked_Technology_Dataset_200_Observations(5).csv")

print("Dataset loaded successfully")
print(df.shape)

Dataset loaded successfully
(200, 11)


In [12]:
# Correct Analog Devices board independence data

df.loc[(df["Company"] == "Analog Devices") & (df["Year"] == 2015),
       ["Independent Directors (N)", "Board Independence (%)"]] = [9, 81.818182]

df.loc[(df["Company"] == "Analog Devices") & (df["Year"] == 2016),
       ["Independent Directors (N)", "Board Independence (%)"]] = [8, 80.0]

# Check corrected rows
df.loc[
    (df["Company"] == "Analog Devices") &
    (df["Year"].isin([2015, 2016])),
    ["Company", "Year", "Board Size",
     "Independent Directors (N)", "Board Independence (%)"]
]

,Company,Year,Board Size,Independent Directors (N),Board Independence (%)
26,Analog Devices,2015,11,9,81.818182
27,Analog Devices,2016,10,8,80.000000


In [13]:
variables = [
    "ROA (%)",
    "Female Directors (%)",
    "Board Size",
    "Board Independence (%)",
    "Firm Size (ln assets)",
    "Leverage (%)"
]

descriptive_stats = df[variables].describe().T

descriptive_stats = descriptive_stats[
    ["count", "mean", "std", "min", "max"]
]

descriptive_stats.round(3)

,count,mean,std,min,max
ROA (%),200.0,14.103,7.812,-12.571,35.165
Female Directors (%),200.0,17.237,9.440,0.000,44.444
Board Size,200.0,10.405,1.737,6.000,15.000
Board Independence (%),200.0,71.682,9.708,33.333,100.000
Firm Size (ln assets),200.0,9.785,1.217,7.577,11.668
Leverage (%),200.0,42.963,15.729,7.551,91.749


####  Correlation Analysis

In [14]:
correlation_matrix = df[
    [
        "ROA (%)",
        "Female Directors (%)",
        "Board Size",
        "Board Independence (%)",
        "Firm Size (ln assets)",
        "Leverage (%)"
    ]
].corr()

correlation_matrix.round(3)

,ROA (%),Female Directors (%),Board Size,Board Independence (%),Firm Size (ln assets),Leverage (%)
ROA (%),1.000,0.357,-0.110,0.042,0.124,-0.005
Female Directors (%),0.357,1.000,0.224,0.061,0.230,0.017
Board Size,-0.110,0.224,1.000,0.247,0.147,0.011
Board Independence (%),0.042,0.061,0.247,1.000,0.120,0.156
Firm Size (ln assets),0.124,0.230,0.147,0.120,1.000,0.289
Leverage (%),-0.005,0.017,0.011,0.156,0.289,1.000


####  Multiple Linear Regression

In [15]:
X = df[
    [
        "Female Directors (%)",
        "Board Size",
        "Board Independence (%)",
        "Firm Size (ln assets)",
        "Leverage (%)"
    ]
]

y = df["ROA (%)"]

# Add constant/intercept
X = sm.add_constant(X)

# Run OLS regression
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                ROA (%)   R-squared:                       0.175
Model:                            OLS   Adj. R-squared:                  0.154
Method:                 Least Squares   F-statistic:                     8.239
Date:                Thu, 13 Aug 2026   Prob (F-statistic):           4.46e-07
Time:                        03:50:23   Log-Likelihood:                -675.15
No. Observations:                 200   AIC:                             1362.
Df Residuals:                     194   BIC:                             1382.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     11

In [16]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# VIF test
vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif_data.round(3)

,Variable,VIF
0,const,120.875
1,Female Directors (%),1.101
2,Board Size,1.127
3,Board Independence (%),1.095
4,Firm Size (ln assets),1.170
5,Leverage (%),1.116


In [17]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Breusch-Pagan test
bp_test = het_breuschpagan(model.resid, model.model.exog)

labels = [
    "LM Statistic",
    "LM p-value",
    "F Statistic",
    "F p-value"
]

bp_results = dict(zip(labels, bp_test))

for key, value in bp_results.items():
    print(f"{key}: {value:.6f}")

LM Statistic: 19.484701
LM p-value: 0.001561
F Statistic: 4.188046
F p-value: 0.001230


In [18]:
from scipy.stats import shapiro

# Shapiro-Wilk normality test on regression residuals
shapiro_stat, shapiro_p = shapiro(model.resid)

print(f"Shapiro-Wilk Statistic: {shapiro_stat:.6f}")
print(f"p-value: {shapiro_p:.6f}")

Shapiro-Wilk Statistic: 0.968299
p-value: 0.000175


####  OLS regression with robust standard errors (HC3)

In [19]:
robust_model = model.get_robustcov_results(cov_type='HC3')

print(robust_model.summary())

                            OLS Regression Results                            
Dep. Variable:                ROA (%)   R-squared:                       0.175
Model:                            OLS   Adj. R-squared:                  0.154
Method:                 Least Squares   F-statistic:                     7.326
Date:                Thu, 13 Aug 2026   Prob (F-statistic):           2.60e-06
Time:                        03:50:23   Log-Likelihood:                -675.15
No. Observations:                 200   AIC:                             1362.
Df Residuals:                     194   BIC:                             1382.
Df Model:                           5                                         
Covariance Type:                  HC3                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
const                     11

In [20]:
from statsmodels.stats.stattools import durbin_watson

# Durbin-Watson test for autocorrelation
dw_stat = durbin_watson(model.resid)

print(f"Durbin-Watson Statistic: {dw_stat:.3f}")

Durbin-Watson Statistic: 1.025


#### Analysis Summary

The analysis examined the relationship between board diversity and firm performance using a balanced panel dataset of 200 observations from 20 technology companies over the period 2009–2018. Descriptive statistics, correlation analysis and multiple linear regression were conducted. Diagnostic tests were also performed to assess the regression assumptions. As heteroscedasticity was identified, HC3 robust standard errors were used for the final regression model. The results indicate that female board representation has a significant positive relationship with ROA, while board size has a significant negative relationship. Board independence, firm size and leverage were not statistically significant.